# Wolfenstein 3D Map Data — Research Notebook

Working reference notes on the original Wolfenstein 3D map data format,
tile/thing value tables, and a comparison color legend from an existing
open-source renderer. This notebook exists to become a **programming
and user reference** for the `wolf3dmaps` tile-extraction effort — it is
grown incrementally as research is done, rather than executed for
computation.

Companion doc: [devlog_20260725.md](devlog_20260725.md)

**Reference source clone:** the original id Software source release,
[id-Software/wolf3d](https://github.com/id-Software/wolf3d), is cloned
locally for reference at `~/Agon/id/wolf3d` — kept outside this
repository and never vendored/copied in, since it ships under id's own
Limited Use Software License Agreement rather than a permissive
open-source license. See the devlog's "Reference source code" section
for the licensing rationale.

## 1. Level structure

- Each level is a **64x64 grid** of cells (confirmed fixed size — the
  original engine hard-codes 64x64 and ignores the width/height fields
  stored in the level header).
- Level data is split across up to three same-sized 64x64 "planes",
  each cell being a `UInt16LE`:
  - **Plane 0** — structural: walls, doors, elevators, area/room tiles.
  - **Plane 1** — "things": player spawn, enemies, collectibles,
    decorative props, secret pushwalls.
  - **Plane 2** — unused by Wolfenstein 3D itself (present in the file
    format for other id Software games built on the same engine).
- Values are stored row-major, left-to-right then top-to-bottom.

**Sources:**
- [Wolfenstein 3D game file specifications (vpoupet)](https://vpoupet.github.io/wolfenstein/docs/files) —
  detailed, source-code-backed breakdown of `MAPHEAD.WL6`,
  `GAMEMAPS.WL6`, and `VSWAP.WL6`, itself citing Gaarabis's original
  spec and the Wolfenstein Fandom wiki.
- [id Software's original released source (WL_DEF.H)](https://github.com/id-Software/wolf3d/blob/master/WOLFSRC/WL_DEF.H) —
  confirms `MAPSIZE 64`, `MAXWALLTILES 64`, `AREATILE 107`,
  `NUMAREAS 37`, `ELEVATORTILE 21`, `PUSHABLETILE 98`, `EXITTILE 99`.
- [GameMaps Format (ModdingWiki)](https://moddingwiki.shikadi.net/wiki/GameMaps_Format) —
  byte-level layout of `MAPHEAD`/`GAMEMAPS`, RLEW + Carmack compression.

## 2. Plane 0 — structural tile values

| Value range | Meaning |
| --- | --- |
| `000`–`063` | Wall (value selects the wall texture pair used by the renderer) |
| `021` | Elevator wall specifically (`ELEVATORTILE`) |
| `090`–`091` | Regular unlocked door (oriented: N/S vs E/W) |
| `092`–`093` | Gold-locked door (oriented) |
| `094`–`095` | Silver-locked door (oriented) |
| `100`–`101` | Elevator door (oriented) |
| `106` | Ambush tile (`AMBUSHTILE`) — a walkable/room tile that silently alerts enemies |
| `107`–`143` | Walkable floor tile, one value per "area"/room (`AREATILE` + `NUMAREAS` = 37 distinct areas) — used for enemy AI room-connectivity, not textures |

Notes:
- A cell is only rendered as a wall in the overview-style renderer if
  its Plane 0 value is `<= 63` **and** it's adjacent (incl. diagonally)
  to another non-wall cell — i.e. "connected" walls are drawn, isolated
  interior wall values are not (see `connectedWall()` in section 4).
- Secret push-walls are **not** a distinct Plane 0 value — a normal
  wall tile (`<= 63`) becomes a pushwall when its Plane 1 value is `98`
  (`PUSHABLETILE`).

## 3. Plane 1 — "things" (actors, items, spawn, pushwalls)

| Value range | Meaning |
| --- | --- |
| `019`–`022` | Player spawn point (oriented, one value per facing direction) |
| `023`–`070` | Decorative/collectible props (see prop table below) |
| `043`–`044` | Key (silver, gold) |
| `047` | Food |
| `048` | Medkit |
| `049` | Ammo clip |
| `050` | Machine gun |
| `051` | Chaingun |
| `052`–`055` | Treasure (cross, chalice, chest, crown) |
| `056` | Extra life |
| `098` | Secret pushwall marker (paired with a wall tile in Plane 0) |
| `108`–`227` | Enemies (guard/officer/SS/dog/mutant, oriented, by difficulty + patrol state; named bosses/specials at fixed single values, see below) |
| `124` | Dead guard (decorative) |

### Selected blocking props (23–70 range)

| Value | Prop |
| --- | --- |
| `24` | Green barrel |
| `25` | Table and chairs |
| `26` | Floor lamp |
| `28` | Hanged skeleton |
| `29` | Dog food |
| `30` | White pillar |
| `31` | Tree |
| `33` | Sink |
| `34` | Potted plant |
| `35` | Urn |
| `36` | Bare table |
| `39` | Suit of armor |
| `40` | Hanging cage |
| `41` | Skeleton in cage |
| `45` | Bed |
| `58` | Barrel |
| `59` | Well |
| `60` | Empty well |
| `62` | Flag |
| `63` | "Call Apogee" sign |
| `68` | Stove |
| `69` | Spears |

### Named bosses / unique enemies (always facing player, all difficulties)

| Value | Enemy |
| --- | --- |
| `160` | Fake Hitler |
| `178` | Hitler |
| `179` | General Fettgesicht |
| `196` | Doctor Schabbs |
| `197` | Gretel Grosse |
| `214` | Hans Grosse |
| `215` | Otto Giftmacher |
| `224` | Blinky (red ghost) |
| `225` | Pinky (pink ghost) |
| `226` | Clyde (orange ghost) |
| `227` | Inky (blue ghost) |

Regular enemy types (guard, officer, SS, dog, mutant) each occupy 24
consecutive values per type (4 facing directions x standing/patrolling
x 3 difficulty tiers) — full ranges are in the source spec linked
above if/when needed for actor extraction.

## 4. Comparison: an existing overview-map color legend

[vpoupet/wolfenstein](https://github.com/vpoupet/wolfenstein)'s
`js/map.js` renders a HUD overview map from live Plane 0 / Plane 1 data
— conceptually the same kind of image as our
[Episode_1/0.png](../src/assets/maps/orig/Episode_1/0.png) source
files. Its color choices are a useful reference point (not necessarily
identical to whatever produced our source images, but a good starting
hypothesis for classifying pixel colors):

| Color | Meaning |
| --- | --- |
| `#000000` (black) | Wall (connected wall tile, Plane 0 <= 63) |
| `#FFFFFF` (white) | Regular door |
| `#FFFF00` (yellow) | Gold-locked door |
| `#00FFFF` (cyan) | Silver-locked door |
| `#FF8F00` (orange) | Elevator |
| `#FF0000` (red) | Secret pushwall |
| `#888888` (grey, background fill) | Floor / room tile |
| `#444444` (dark grey, small dot) | Blocking prop/obstacle |
| small dot on `#888888` | Treasure, keys (drawn as small inset squares) |
| `#00FF00` (green) | Player position |

Relevant logic (`connectedWall`): a Plane-0 wall tile is only drawn as
a wall pixel if at least one of its 8 neighbors (orthogonal + diagonal)
is *not* a wall — i.e., solid interior rock is left undrawn/blank,
only room-facing wall surfaces are rendered. This is a good candidate
rule to test against our own source images if they turn out to follow
similar conventions.

## 5. Level names (per episode/floor)

The original id Software source release includes `WOLFSRC/MAPSWL6.H`,
a TED5 (the original level editor) map-name header. It's a plain C
`enum` — genuinely both human- and machine-readable — listing all 60
level slots in order:

```c
typedef enum
{
    WOLF1_MAP1_MAP,   // 0
    WOLF1_MAP2_MAP,   // 1
    ...
    WOLF1_MAP7_MAP,   // 6
    WOLF1_MAP8_MAP,   // 7
    WOLF1_BOSS_MAP,   // 8
    WOLF1_SECRET_MAP, // 9
    WOLF2_MAP1_MAP,   // 10
    ...
} mapnames;
```

This confirms, straight from source, the structure we already adopted
for [src/assets/maps/orig](../src/assets/maps/orig): **10 floors per
episode** (8 regular floors, then a boss floor, then a secret floor),
6 episodes, 60 levels total — lining up with our zero-based `0.png`
`.9.png` renaming (index `8` = boss floor, index `9` = secret floor).

Source: [MAPSWL6.H](https://raw.githubusercontent.com/id-Software/wolf3d/master/WOLFSRC/MAPSWL6.H)
in [id-Software/wolf3d](https://github.com/id-Software/wolf3d).

Note: this header only names the 60 level *slots* — it is not the tile
data itself. The actual per-level tile/thing grids only exist as
compressed binary in `GAMEMAPS.WL6`/`MAPHEAD.WL6` (see below).

## 6. On machine/human-readable map data in general

Checked the most likely places for an existing decoded/human-readable
version of the actual level *tile grids* (as opposed to just level
names):

- [id-Software/wolf3d](https://github.com/id-Software/wolf3d) (original
  source release) — only ships C/ASM source and the level-*name* enum
  above; no decoded map data.
- [vpoupet/wolfenstein](https://github.com/vpoupet/wolfenstein)'s
  `data/` directory — only the original binary `GAMEMAPS.WL6`,
  `MAPHEAD.WL6`, `VSWAP.WL6` files (RLEW + Carmack compressed), decoded
  at runtime in JS; no pre-decoded JSON/text export found in the repo.

**Conclusion:** no existing machine-readable (e.g. JSON) per-tile map
definition was found anywhere searched so far — every source ships the
original compressed binary format and decodes it on the fly. This
reinforces that extracting our own machine-readable format directly
from the overview images (this project's actual goal) isn't
duplicating existing work.

## 7. Open items / still to research

- [ ] **Named wall textures.** We have confirmed *ranges* (0–63 = wall)
      and engine constants, but not a definitive mapping of individual
      wall values (e.g. `1`, `3`, `5`, ...) to human-readable texture
      names (e.g. "grey stone", "wood door surround", "blue SS
      insignia wall"). This requires correlating `VSWAP.WL6` wall
      texture chunk order with a reference image set — no
      authoritative numbered list was found in this research pass.
      Revisit if/when per-wall-type classification (beyond "is a
      wall") is needed.
- [ ] **Our images' actual color legend.** The above table is from a
      *different* project's renderer, not necessarily how
      `src/assets/maps/orig` images were generated. Needs empirical
      verification by sampling pixel colors from
      [Episode_1/0.png](../src/assets/maps/orig/Episode_1/0.png).
- [ ] Full enemy-value tables (per-direction, per-difficulty ranges)
      only summarized here; expand inline if/when actor extraction
      becomes part of the schema.